# Qwen Taboo J-Lens: environment smoke test

**Objective:** verify the persistent RunPod kernel, GPU runtime, package versions, project paths, and small Hugging Face metadata before downloading Qwen3.6-27B weights.

**Success criteria:** CUDA is available on an approximately 80 GB GPU; the environment report is saved; and the model, adapter, and exact `_n1000` J-Lens metadata pass `scripts/verify_artifacts.py`. This notebook does not load the 27B model.


In [ ]:
from __future__ import annotations

import json
import random
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEED = 7
random.seed(SEED)
PROJECT_ROOT


## Plan

1. Record the environment and GPU.
2. Confirm the declared smoke-test condition.
3. Run metadata-only Hugging Face checks.
4. Stop for review before downloading large weights.


In [ ]:
from src.environment_report import save_environment

environment = save_environment(PROJECT_ROOT / "results/environment_report.json")
{
    "python": environment["python"].split()[0],
    "packages": environment["packages"],
    "torch_runtime": environment.get("torch_runtime"),
    "torch_runtime_error": environment.get("torch_runtime_error"),
}


In [ ]:
runtime = environment.get("torch_runtime", {})
assert runtime.get("cuda_available"), "CUDA is unavailable; stop before downloading weights."
gpu_gib = [round(device["total_memory_bytes"] / 2**30, 1) for device in runtime.get("devices", [])]
print({"devices": runtime.get("devices"), "memory_gib": gpu_gib})
if not any(memory >= 75 for memory in gpu_gib):
    print("WARNING: no approximately 80 GB GPU detected. Do not silently quantize or CPU-offload.")


In [ ]:
config_path = PROJECT_ROOT / "configs/smoke_test.json"
config = json.loads(config_path.read_text())
config


In [ ]:
completed = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "scripts/verify_artifacts.py")],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError("Artifact preflight failed; inspect results/artifact_preflight.json")
artifact_report = json.loads((PROJECT_ROOT / "results/artifact_preflight.json").read_text())
artifact_report


## Review gate

Inspect `results/environment_report.json` and `results/artifact_preflight.json`. Record resolved SHAs in `research_log.md`. Do not download the 27B weights until the GPU, adapter base, tokenizer/base revision, and exact `_n1000` lens have been reviewed.

If MCP cannot list, open, and execute this notebook within 30–45 focused minutes, keep Jupyter for human inspection and run experiments as scripts through remote Codex/SSH instead.
